# Fabric migration (v2) – step 1: check and connect workspaces to Azure DevOps

> **AS-IS - independent personal/community project.**
> These notebooks are provided **AS-IS**, without warranty of any kind, express or implied.
> They are a personal/community contribution, not a Microsoft product or service, and are not published on
> behalf of, endorsed, supported or warranted by Microsoft. They do not represent or bind Microsoft and
> create no obligation or commitment by Microsoft. Product names, APIs and documentation are technical
> references only. Use at your own risk and validate in your own environment before production or deletion.

For each workspace: reports its Git connection (the same link as *Workspace settings > Git integration*), and
links the ones that aren't linked to the **same repository and branch** (`ADO_BRANCH`), each in its own folder,
then makes the first commit (workspace → Git). A workspace linked earlier but never synced is finished.

**Why linking can fail, and what this notebook does about it**
| Cause | Check / fix |
|---|---|
| You're Fabric administrator but not **Admin of the workspace** | Shows your role per workspace; `ADD_ME_AS_WORKSPACE_ADMIN = True` adds you (admin API) |
| Fabric has no Azure DevOps credentials for you | Open any workspace > Source control once and sign in, or use a stored connection (`GIT_CONNECTION_ID`) |
| Notebook runs as a service principal / workspace identity (scheduled or pipeline run) | Refused unless `GIT_CONNECTION_ID` is set |
| Anything else | Each failure prints the error and the fix; the workspaces that couldn't be linked are listed with the values to enter by hand |

**Safeguards**
- Dry run first: prints the plan and a CONNECT ID; a real run needs `CONFIRM_CONNECT_ID`.
- Workspaces already connected are never changed; connections to another organization, project or repository are
  reported for review.
- Connects only to an **empty** Git folder, so the first sync can only go from the workspace to Git, never the other
  way (nothing in the workspace can be overwritten). Two workspaces can't share a folder.
- Stops a workspace if Fabric asks to update the workspace from Git instead of committing.
- At the end, lists the Fabric items that are still not in Git (item types Git doesn't support): they rely on the
  ADLS copies.

**Requirements**
- Admin role on each workspace (or `ADD_ME_AS_WORKSPACE_ADMIN`); the workspace on a Fabric capacity.
- Your Git credentials for Azure DevOps (SSO) or a Fabric connection in `GIT_CONNECTION_ID`; *Contribute* on the
  branch. Use a branch without pull-request policies, or the commit is refused.
- Azure DevOps token in Key Vault (*Code: Read & write*) to check folders and create a missing branch.

In [ ]:
WORKSPACE_IDS = [
    # "00000000-0000-0000-0000-000000000000",
]

ADO_ORGANIZATION = "<devops-organization>"
ADO_PROJECT = "<devops-project>"
ADO_REPOSITORY = "<repository-name>"
ADO_BRANCH = "fabric-workspaces"         # created from the default branch if it doesn't exist
CREATE_BRANCH_IF_MISSING = True
# Git folder per workspace. {workspace} = workspace name (made path-safe), {id} = workspace ID.
DIRECTORY_PATTERN = "/workspaces/{workspace}"
DIRECTORY_MAP = {}                        # optional per-workspace folder: {"workspace-guid": "/path"}

KEY_VAULT_URL = "https://<key-vault-name>.vault.azure.net/"
ADO_AUTH = "pat"                         # how notebooks sign in to Azure DevOps: "pat" or "service_principal"
ADO_PAT_SECRET_NAME = "<secret-name>"    # pat: Key Vault secret holding the personal access token
ADO_SP_TENANT_ID = ""                    # service_principal: Entra tenant ID (GUID) of the app registration
ADO_SP_CLIENT_ID = ""                    # service_principal: application (client) ID
ADO_SP_SECRET_NAME = ""                  # service_principal: Key Vault secret holding the client secret VALUE

# ---------------------------------------------------------------- how Fabric signs in to Azure DevOps for the Git link
# (separate from ADO_AUTH, which is only how this notebook itself calls DevOps)
#   "service_principal" = Fabric connection "Azure DevOps - Source control" with the service principal in ADO_SP_*
#                         (fill those even if ADO_AUTH = "pat"). Found by GIT_CONNECTION_NAME, or created when you
#                         confirm the plan. Recommended: not tied to one person, works when the notebook runs as a
#                         service principal. The service principal needs Basic access + project Contributors.
#   "my_account"        = your own Azure DevOps sign-in (you must be a member of the DevOps organization).
#   "connection_id"     = an existing Fabric connection: put its ID in GIT_CONNECTION_ID.
GIT_LINK_AUTH = "service_principal"
GIT_CONNECTION_NAME = ""                  # "" = fabric-migration-<organization>-<project>-<repository>
GIT_CONNECTION_ID = ""                    # "connection_id": its ID (with "service_principal": force this one)

# Your own Git sign-in per workspace (what the workspace screens and every later Git call use). False (default) =
# the notebook only reports it; you choose the connection yourself in the workspace: Source control > account >
# the Fabric connection used to link (this is what fixed "Can't fetch branches" in testing). True = the notebook
# switches it for you through the API (part of the confirmed plan; not proven to fix the screens).
SWITCH_MY_GIT_CREDENTIALS = False

# This migration keeps Fabric, Azure DevOps and the service principal in ONE Entra tenant. The notebook stops if
# the DevOps organization (or the service principal) belongs to another tenant. True only for a future cross-tenant
# scenario; not part of this migration.
ALLOW_CROSS_TENANT = False

# Linking needs the Admin role on each workspace (Fabric administrator isn't enough). True = workspaces where you
# aren't Admin get you added as Admin first (admin API, part of the confirmed plan). Remove it after the migration.
ADD_ME_AS_WORKSPACE_ADMIN = False

DRY_RUN = True
CONFIRM_CONNECT_ID = ""

## Helpers
Same helpers as the snapshot notebook. Nothing runs here.

In [ ]:
BUILD_ID = "0a17a04eabd1"
print("V2_1_Connect_Workspaces_To_Git build 0a17a04eabd1 (compare with BUILD_IDS.txt of the package you meant to run)")
# AS-IS, without warranty of any kind; independent personal/community code.
# Not a Microsoft product or service; not published on behalf of Microsoft.
# Does not represent or bind Microsoft; no Microsoft endorsement, warranty or support commitment.
"""Credential redaction for diagnostic output, never for API or backup payloads."""
import re
from collections.abc import Mapping
from typing import TypeVar, overload
from urllib.parse import quote

REDACTED = "<redacted>"
if "_REDACTION_VALUES" not in globals():
    _REDACTION_VALUES: set[str] = set()

_REDACTION_KEYS = {
    "authorization", "accesstoken", "refreshtoken", "clientsecret", "serviceprincipalsecret",
    "password", "passwd", "accountkey", "apikey", "personalaccesstoken", "pat", "secret", "token",
    "connectionstring", "sastoken", "sharedaccesssignature", "sig",
}
_REDACTION_PATTERNS = (
    re.compile(r"\bgh[pousr]_[A-Za-z0-9]{20,}\b"),
    re.compile(r"\bgithub_pat_[A-Za-z0-9_]{40,}\b"),
    re.compile(r"\beyJ[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\.[A-Za-z0-9_-]{16,}\b"),
    re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----.*?"
               r"-----END (?:RSA |EC |OPENSSH )?PRIVATE KEY-----", re.DOTALL),
)
_REDACTION_AUTH = re.compile(r"\b(Bearer|Basic)([ \t]+)[A-Za-z0-9_+/=.-]{16,}", re.IGNORECASE)
_REDACTION_QUERY = re.compile(
    r"([?&](?:sig|access_token|refresh_token|client_secret|password|api_key)=)[^&\s\"'<>]+",
    re.IGNORECASE,
)
_REDACTION_ASSIGNMENT = re.compile(
    r"""(\b(?:client[_-]?secret|servicePrincipalSecret|access[_-]?token|refresh[_-]?token|
    password|passwd|account[_-]?key|api[_-]?key|personalAccessToken|sas[_-]?token|pat|secret|token)
    \b["']?\s*[:=]\s*["']?)([^\s"'&,;}\]<>]+)""",
    re.IGNORECASE | re.VERBOSE,
)


def register_sensitive_value(value: str) -> None:
    if len(value) >= 8:
        _REDACTION_VALUES.add(value)


def redact_sensitive_text(text: str) -> str:
    for value in sorted(_REDACTION_VALUES.copy(), key=len, reverse=True):
        text = text.replace(value, REDACTED).replace(quote(value, safe=""), REDACTED)
    for pattern in _REDACTION_PATTERNS:
        text = pattern.sub(REDACTED, text)
    text = _REDACTION_AUTH.sub(lambda m: m[1] + m[2] + REDACTED, text)
    text = _REDACTION_QUERY.sub(lambda m: m[1] + REDACTED, text)
    return _REDACTION_ASSIGNMENT.sub(lambda m: m[1] + REDACTED, text)


_Key = TypeVar("_Key")


@overload
def redact_sensitive_data(value: str) -> str: ...


@overload
def redact_sensitive_data(value: Mapping[_Key, object]) -> dict[_Key, object]: ...


@overload
def redact_sensitive_data(value: list[object]) -> list[object]: ...


@overload
def redact_sensitive_data(value: tuple[object, ...]) -> tuple[object, ...]: ...


@overload
def redact_sensitive_data(value: object) -> object: ...


def redact_sensitive_data(value: object) -> object:
    if isinstance(value, Mapping):
        out = {}
        for key, item in value.items():
            normalized = re.sub(r"[^a-z0-9]", "", str(key).lower())
            if normalized in _REDACTION_KEYS and item is not None and not (isinstance(item, str) and item == ""):
                out[key] = REDACTED
            else:
                out[key] = redact_sensitive_data(item)
        return out
    if isinstance(value, list):
        return [redact_sensitive_data(item) for item in value]
    if isinstance(value, tuple):
        return tuple(redact_sensitive_data(item) for item in value)
    return redact_sensitive_text(value) if isinstance(value, str) else value
import base64, datetime as dt, hashlib, json, re, threading, time, uuid
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote

import pandas as pd

pd.set_option("display.max_colwidth", None)   # never cut error messages or fixes in tables
import requests

try:
    notebookutils  # preloaded in Fabric notebooks
except NameError:
    import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
ONELAKE_DFS = "https://onelake.dfs.fabric.microsoft.com"
STORAGE_VERSION = "2023-11-03"
MAX_ATTEMPTS = 6
MAX_RETRY_WAIT_SECONDS = 15 * 60
LRO_TIMEOUT_SECONDS = 60 * 60
CHUNK_BYTES = 8 * 1024 * 1024
ZERO_GUID = "00000000-0000-0000-0000-000000000000"
GUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
NOW_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

RESULTS = []           # one row per item and step: this notebook's log
_lock = threading.Lock()
CALLS, RETRIES = Counter(), Counter()


def norm_id(value):
    text = str(value or "").strip().strip("{}").lower()
    return text if GUID_RE.match(text) else None


def safe_name(name, max_len=80):
    """A folder-safe version of a Fabric display name."""
    text = re.sub(r'[\\/:*?"<>|#%\x00-\x1f]', "_", str(name or "")).strip().strip(".")
    return (text[:max_len].rstrip(" .") or "unnamed")


def log(workspace, item, step, status, detail=""):
    """One row of this notebook's log. workspace: name; item: dict with type/displayName/id (or {})."""
    detail = redact_sensitive_text(str(detail))
    with _lock:
        RESULTS.append({"Workspace": workspace, "Item": item.get("displayName", ""), "Type": item.get("type", ""),
                        "ItemId": item.get("id", ""), "Step": step, "Status": status,
                        "Detail": str(detail)})
    if status not in ("OK", "Dry run", "Skipped", "Unchanged", "Info", "Ready"):
        print(f"  ! {workspace} / {item.get('type', '')} {item.get('displayName', '')} / {step}: {status} - "
              f"{detail}")


def show(df, rows=200):
    try:
        display(df.head(rows))
    except NameError:
        print(df.head(rows).to_string())


def human(n):
    n = float(n or 0)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


class HttpError(Exception):
    def __init__(self, status, code, message, url):
        self.status, self.code, self.message, self.url = status, code or "", redact_sensitive_text(str(message or "")), url
        super().__init__(self.short())

    def short(self):
        head = f"HTTP {self.status} {self.code}".strip() if self.status else str(self.code)
        return f"{head}: {self.message}" if self.message else head


RAW_BODY_LIMIT = 4000   # only for non-JSON bodies (for example a whole HTML page); JSON error messages are kept whole


def _raw_text(text):
    text = re.sub(r"<[^>]+>", " ", text or "")              # HTML page -> its text
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= RAW_BODY_LIMIT:
        return text
    return text[:RAW_BODY_LIMIT] + f" ... [{len(text) - RAW_BODY_LIMIT} more characters of the response body]"


def _error_details(resp):
    try:
        body = resp.json()
    except ValueError:
        return "", _raw_text(resp.text)
    if not isinstance(body, dict):
        return "", json.dumps(body)
    err = body.get("error") if isinstance(body.get("error"), dict) else body
    code = err.get("errorCode") or err.get("code") or err.get("typeKey") or err.get("error") or ""
    msg = err.get("message") or err.get("@message") or err.get("error_description") or ""
    if not msg:
        msg = json.dumps(body)                                  # unknown error shape: keep all of it
    return str(code), str(msg)


def _retry_after(resp, attempt):
    try:
        return max(1, int(float(resp.headers.get("Retry-After", ""))))
    except (TypeError, ValueError):
        return min(300, 10 * 2 ** (attempt - 1))


_local = threading.local()


def _session():
    if getattr(_local, "session", None) is None:
        _local.session = requests.Session()
    return _local.session


def http(method, url, audience=None, *, auth=None, headers=None, params=None, json_body=None, data=None,
         timeout=300, retry=True, label="other"):
    """One HTTP call with a fresh Entra token. Retries 408/429/5xx (honouring Retry-After) and network errors."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        hdrs = dict(headers or {})
        if auth:
            hdrs["Authorization"] = auth
        elif audience:
            hdrs["Authorization"] = "Bearer " + notebookutils.credentials.getToken(audience)
        if hdrs.get("Authorization"):
            register_sensitive_value(hdrs["Authorization"].split(" ", 1)[-1])
        try:
            resp = _session().request(method, url, params=params, json=json_body, data=data,
                                      headers=hdrs, timeout=timeout)
        except requests.RequestException as exc:
            if not retry or attempt == MAX_ATTEMPTS:
                raise HttpError(None, type(exc).__name__, str(exc), url)
            time.sleep(min(60, 5 * attempt))
            continue
        with _lock:
            CALLS[label] += 1
        if retry and (resp.status_code in (408, 429) or resp.status_code >= 500):
            wait = _retry_after(resp, attempt)
            if attempt < MAX_ATTEMPTS and wait <= MAX_RETRY_WAIT_SECONDS:
                with _lock:
                    RETRIES[label] += 1
                time.sleep(wait)
                continue
        if resp.status_code >= 400:
            raise HttpError(resp.status_code, *_error_details(resp), url)
        return resp


# ---------------------------------------------------------------- Fabric REST API
def _wait_operation(resp, want_result=True):
    """Follow a Fabric long-running operation (202 Accepted) to its result."""
    location = resp.headers.get("Location")
    op_id = resp.headers.get("x-ms-operation-id")
    if not location and op_id:
        location = f"{FABRIC_API}/operations/{op_id}"
    if not location:
        return {}
    wait = _retry_after(resp, 1) if resp.headers.get("Retry-After") else 2
    deadline = time.monotonic() + LRO_TIMEOUT_SECONDS
    while True:
        time.sleep(min(max(wait, 1), 30))
        state = http("GET", location, "pbi", label="fabric").json()
        status = str(state.get("status", "")).lower()
        if status == "succeeded":
            if not want_result:
                return state
            try:
                result = http("GET", location.rstrip("/") + "/result", "pbi", label="fabric")
            except HttpError as exc:
                if exc.status in (400, 404):      # operations without a result (commit, update from Git)
                    return state
                raise
            return result.json() if result.content else {}
        if status in ("failed", "cancelled", "canceled"):
            err = state.get("error") or {}
            raise HttpError(None, err.get("errorCode") or status, err.get("message"), location)
        if time.monotonic() > deadline:
            raise HttpError(None, "Timeout", "the long-running operation didn't finish", location)
        wait = 3


def fabric(method, path, body=None, params=None, want_result=True):
    resp = http(method, FABRIC_API + path, "pbi", json_body=body, params=params, label="fabric")
    if resp.status_code == 202:
        return _wait_operation(resp, want_result)
    return resp.json() if resp.content else {}


def fabric_pages(path, key="value", params=None):
    url, prm, seen = FABRIC_API + path, dict(params or {}), set()
    while url:
        body = http("GET", url, "pbi", params=prm or None, label="fabric").json()
        yield from body.get(key) or []
        if body.get("continuationUri"):
            url, prm = body["continuationUri"], {}
        elif body.get("continuationToken"):
            url, prm = FABRIC_API + path, {**(params or {}), "continuationToken": body["continuationToken"]}
        else:
            break
        marker = (url, json.dumps(prm, sort_keys=True))
        if marker in seen:
            break
        seen.add(marker)


# ---------------------------------------------------------------- ADLS Gen2 (secondary backup + migration records)
ADLS_HEADERS = {"x-ms-version": STORAGE_VERSION}


def adls_url(path):
    return f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}/{quote(path.strip('/'), safe='/')}"


def adls_create(path):
    http("PUT", adls_url(path), "storage", headers=ADLS_HEADERS, params={"resource": "file"}, label="adls")


def adls_append(path, position, data):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=data,
         params={"action": "append", "position": str(position)}, label="adls")


def adls_flush(path, position):
    http("PATCH", adls_url(path), "storage", headers=ADLS_HEADERS, data=b"",
         params={"action": "flush", "position": str(position)}, label="adls")


def adls_write(path, data):
    """Write a whole (small) file, replacing any existing one."""
    if isinstance(data, str):
        data = data.encode("utf-8")
    adls_create(path)
    for pos in range(0, len(data), CHUNK_BYTES):
        adls_append(path, pos, data[pos:pos + CHUNK_BYTES])
    adls_flush(path, len(data))
    return len(data)


def adls_list(path, recursive=True):
    """[(path relative to `path`, size, is_dir)] under a folder of the container; [] if it doesn't exist."""
    url = f"https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}"
    params = {"resource": "filesystem", "recursive": str(recursive).lower(), "directory": path.strip("/")}
    out, prefix = [], path.strip("/") + "/"
    while True:
        try:
            resp = http("GET", url, "storage", headers=ADLS_HEADERS, params=params, label="adls")
        except HttpError as exc:
            if exc.status == 404:
                return out
            raise
        for e in resp.json().get("paths", []):
            name = e["name"][len(prefix):] if e["name"].startswith(prefix) else e["name"]
            out.append((name, int(e.get("contentLength") or 0), str(e.get("isDirectory", "false")).lower() == "true"))
        token = resp.headers.get("x-ms-continuation")
        if not token:
            return out
        params = {**params, "continuation": token}


def adls_read(path):
    return http("GET", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls").content


def adls_delete(path):
    http("DELETE", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")


def adls_exists(path):
    try:
        http("HEAD", adls_url(path), "storage", headers=ADLS_HEADERS, label="adls")
        return True
    except HttpError as exc:
        if exc.status == 404:
            return False
        raise


# ---------------------------------------------------------------- Power BI item types
# Power BI items survive a cross-region move; every other item type is a Fabric item that must be deleted first.
POWER_BI_TYPES = {"Report", "SemanticModel", "Dashboard", "PaginatedReport"}
# Items Fabric creates and deletes together with their parent; they are never saved in Git on their own.
CHILD_TYPES = {"SQLEndpoint"}
DATA_TYPES = {"Lakehouse": "Data", "Warehouse": "Data", "KQLDatabase": "KQL export"}
# Dataflow Gen2 staging items: Fabric creates them for Dataflows Gen2 (one Lakehouse and one Warehouse per workspace,
# named StagingLakehouseForDataflows_<timestamp> / StagingWarehouseForDataflows_<timestamp>) and creates them again
# when a Dataflow Gen2 runs. Not saved in Git, not backed up, not restored; V2_4 still deletes them (a Fabric item
# left behind blocks a tenant migration).
STAGING_RE = re.compile(r"^Staging(Lakehouse|Warehouse)ForDataflows(_\d+)?$", re.IGNORECASE)


def is_dataflow_staging(item):
    return bool(item.get("staging")) or (item.get("type") in ("Lakehouse", "Warehouse", "SQLEndpoint")
                                         and bool(STAGING_RE.match(str(item.get("displayName") or ""))))
KQL_ADX_STEP = "Kusto Copy"      # V2_2 with KQL_BACKUP_MODE "adx"/"both": the KQL rows are in Azure Data Explorer


def data_step(item_type, status_of):
    """(backup step, status) holding an item's data. KQL databases: the Parquet export if it's OK, otherwise the
    Kusto Copy to ADX if that's OK, otherwise whichever the backup recorded. status_of(step) -> status or ""."""
    step = DATA_TYPES.get(item_type)
    if item_type != "KQLDatabase":
        return step, status_of(step) if step else ""
    exp, adx = status_of(step), status_of(KQL_ADX_STEP)
    if exp == "OK" or not adx:
        return (step, exp) if exp or not adx else (KQL_ADX_STEP, adx)
    if adx == "OK" or not exp:
        return KQL_ADX_STEP, adx
    return step, exp


GUID_ANY_RE = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")


def definition_hash(parts):
    """Fingerprint of an item definition (every part except .platform), to prove it didn't change."""
    h = hashlib.sha256()
    for p in sorted(parts, key=lambda x: x["path"]):
        if p["path"] == ".platform":
            continue
        h.update(p["path"].encode("utf-8") + b"\0" + base64.b64decode(p.get("payload") or "") + b"\0")
    return h.hexdigest()


def adls_only(item):
    """Item in a Git-linked workspace that Git doesn't hold: V2_5 recreates it from its ADLS definition.
    ("secondary backup ..." is the label older snapshots used.)"""
    rec = str(item.get("recovery") or "")
    return not item.get("inGit") and (rec.startswith("ADLS definition (V2_5") or rec.startswith("secondary backup"))


def key_of(item_type, name):
    return f"{item_type}|{str(name).strip().lower()}"


# ---------------------------------------------------------------- Azure DevOps (read the snapshot, manage branches)
ADO_API_VERSION = "7.1"
_ado_auth = {}


ADO_RESOURCE = "499b84ac-1321-427f-aa17-267ca6975798"     # Azure DevOps' fixed Entra application ID


def ado_mode():
    mode = str(globals().get("ADO_AUTH", "pat") or "pat").strip().lower().replace("-", "_").replace(" ", "_")
    if mode in ("sp", "spn", "serviceprincipal"):
        mode = "service_principal"
    if mode not in ("pat", "service_principal"):
        raise ValueError(f"ADO_AUTH = {globals().get('ADO_AUTH')!r}: use \"pat\" or \"service_principal\"")
    return mode


def ado_auth_missing():
    """Settings the chosen ADO_AUTH needs that are still empty or placeholders."""
    need = ["ADO_PAT_SECRET_NAME"] if ado_mode() == "pat" else ["ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME"]
    return [n for n in need if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def _clean_secret(value):
    value = str(value or "").strip().strip('"').strip("'").strip()
    register_sensitive_value(value)
    return value


_sp_auth = {}


def sp_header():
    """Entra token (Bearer header) of the service principal in ADO_SP_* for Azure DevOps, renewed 5 minutes before it
    expires. Used by ADO_AUTH = "service_principal" and to check GIT_LINK_AUTH = "service_principal"."""
    if "header" in _sp_auth and _sp_auth.get("expires", 0) > time.time() + 300:
        return _sp_auth["header"]
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    kind, _ = secret_shape(secret)
    _sp_auth.update(secret_length=len(secret), secret_kind=kind)
    url = f"https://login.microsoftonline.com/{quote(str(ADO_SP_TENANT_ID).strip())}/oauth2/v2.0/token"
    try:
        tok = http("POST", url, data={"grant_type": "client_credentials", "client_id": str(ADO_SP_CLIENT_ID).strip(),
                                      "client_secret": secret, "scope": ADO_RESOURCE + "/.default"},
                   label="entra").json()
    except HttpError as exc:
        raise HttpError(exc.status, exc.code or "SignInFailed",
                        f"service principal sign-in failed: {exc.message} {sp_signin_fix(exc.message, kind)}".strip(),
                        url) from None
    _sp_auth["header"] = "Bearer " + tok["access_token"]
    register_sensitive_value(tok["access_token"])
    _sp_auth["expires"] = time.time() + int(tok.get("expires_in", 3600))
    return _sp_auth["header"]


def ado_header():
    """Authorization header for Azure DevOps: a personal access token, or an Entra token of a service principal
    (renewed 5 minutes before it expires, so long runs keep working). Secrets come from Key Vault and are never shown."""
    if ado_mode() == "pat":
        if "header" not in _ado_auth:
            pat = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_PAT_SECRET_NAME))
            _ado_auth.update(mode="pat", pat_length=len(pat), pat_shape=secret_shape(pat))
            _ado_auth["header"] = "Basic " + base64.b64encode((":" + pat).encode()).decode()
        return _ado_auth["header"]
    _ado_auth["mode"] = "service_principal"
    try:
        header = sp_header()
    finally:
        _ado_auth.update(pat_length=_sp_auth.get("secret_length"), sp_secret_kind=_sp_auth.get("secret_kind"))
    _ado_auth.update(header=header, expires=_sp_auth["expires"])
    return header


def sp_signin_fix(message, kind=""):
    m = str(message or "")
    if kind == "pat":
        return ("The Key Vault secret looks like a personal access token, not a client secret: set ADO_AUTH = \"pat\" "
                "or store the app's client secret value.")
    if "AADSTS7000215" in m or "AADSTS7000222" in m:
        return ("Fix: the client secret is wrong or expired. Store the secret's Value (not its Secret ID) from the app "
                "registration > Certificates & secrets, as a new version of the Key Vault secret.")
    if "AADSTS700016" in m:
        return "Fix: ADO_SP_CLIENT_ID isn't an app in tenant ADO_SP_TENANT_ID: check both IDs."
    if "AADSTS90002" in m or "AADSTS900023" in m:
        return "Fix: ADO_SP_TENANT_ID is wrong: use the Directory (tenant) ID from the app registration's overview."
    return "Check ADO_SP_TENANT_ID, ADO_SP_CLIENT_ID and the client secret value."


def secret_shape(value):
    """(kind, hint) describing what a secret looks like, without revealing it. kind 'pat' = plausible DevOps token."""
    v, n = value, len(value)
    if not v:
        return "empty", "The secret is empty: paste the token as the secret value."
    if re.fullmatch(r"[a-z2-7]{52}", v) or (n == 84 and re.fullmatch(r"[A-Za-z0-9]{84}", v)):
        return "pat", ""
    if re.match(r"^[A-Za-z0-9_\-.~]{3}\dQ~", v) or (n == 40 and "~" in v):
        return "client secret", ("It looks like an Entra ID app registration client secret (40 characters with 'Q~'), "
                                 "not an Azure DevOps personal access token. Create a token in Azure DevOps (User "
                                 "settings > Personal access tokens, scope Code: Read & write) and store that value.")
    if re.fullmatch(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", v):
        return "GUID", ("It is a GUID: probably a client ID, tenant ID or a client secret's 'Secret ID'. None of "
                        "these is an Azure DevOps token.")
    if v.startswith(("ghp_", "gho_", "ghs_", "github_pat_")):
        return "GitHub token", "It looks like a GitHub token, not an Azure DevOps personal access token."
    if re.fullmatch(r"[0-9a-fA-F]{40}", v):
        return "40 hex", ("It is 40 hexadecimal characters: a Git commit ID or an old GitHub token, not an Azure "
                          "DevOps personal access token.")
    if v.startswith(("{", "[")):
        return "JSON", "It holds a JSON value: store only the token itself."
    if v.lower().startswith(("http://", "https://")):
        return "URL", "It holds a URL: store the token, and put the organization in ADO_ORGANIZATION."
    if v.lower().startswith(("bearer ", "basic ")) or re.search(r"\s", v):
        return "several values", "It holds more than one value (spaces or a 'Bearer'/'Basic' prefix): store only the token."
    if v.startswith("eyJ"):
        return "JWT", "It is an Entra access token (JWT), which expires within hours: store a personal access token."
    return "unknown", (f"Azure DevOps tokens have 52 or 84 characters; this has {n}. Check that the secret holds "
                       "the token and nothing else.")


def ado_org_name(value):
    """'contoso', 'https://dev.azure.com/contoso/...' or 'https://contoso.visualstudio.com' -> 'contoso'."""
    text = str(value or "").strip().rstrip("/")
    m = re.match(r"^(?:https?://)?dev\.azure\.com/([^/]+)", text, re.I) or \
        re.match(r"^(?:https?://)?([^./]+)\.visualstudio\.com", text, re.I)
    return m.group(1) if m else text


SIGN_IN_PAGE = ("Azure DevOps answered with its sign-in page instead of data (HTTP 203): it didn't accept the "
                "personal access token for this organization")


SP_NOT_IN_ORG = ("Add the service principal to the organization (Organization settings > Users > Add users, "
                 "search the app name, Basic access) and to the project's Contributors group; the organization must "
                 "be connected to the app's Entra tenant.")


def ado_call(url, method="GET", params=None, body=None, raw=False, auth=None):
    """auth = another Authorization header (for example sp_header()); default: the ADO_AUTH sign-in."""
    sp = auth is not None or _ado_auth.get("mode") == "service_principal"
    try:
        resp = http(method, url, auth=auth or ado_header(), params={"api-version": ADO_API_VERSION, **(params or {})},
                    json_body=body, label="devops", headers={"Accept": "application/octet-stream"} if raw else None)
    except HttpError as exc:
        if exc.status == 401 and (sp or _ado_auth.get("mode") == "service_principal"):
            raise HttpError(401, exc.code or "Unauthorized", f"{exc.message} {SP_NOT_IN_ORG}".strip(), url) from None
        raise
    ctype = resp.headers.get("Content-Type", "json")
    if resp.status_code == 203 or "html" in ctype:
        if sp or _ado_auth.get("mode") == "service_principal":
            raise HttpError(203, "SignInPage", "Azure DevOps answered with its sign-in page instead of data (HTTP "
                            "203): it didn't accept the service principal's token. " + SP_NOT_IN_ORG, url)
        kind, hint = _ado_auth.get("pat_shape", ("pat", ""))
        raise HttpError(203, "SignInPage", SIGN_IN_PAGE + (f". {hint}" if kind != "pat" else ""), url)
    if raw:
        return resp.content
    if "json" not in ctype:
        raise HttpError(resp.status_code, "NotJson", f"Azure DevOps returned {ctype}, not JSON", url)
    return resp.json() if resp.content else {}


def ado(gp, method, path, params=None, body=None, raw=False):
    """Call the Azure DevOps Git API of the repository in a Fabric Git connection (gp = gitProviderDetails)."""
    url = (f"https://dev.azure.com/{quote(ado_org_name(gp['organizationName']))}/{quote(gp['projectName'])}"
           f"/_apis/git/repositories/{quote(gp['repositoryName'])}{path}")
    return ado_call(url, method, params, body, raw)


def ado_ref(gp, branch):
    """{'objectId', 'isLocked'} of a branch, or None if it doesn't exist."""
    refs = ado(gp, "GET", "/refs", params={"filter": f"heads/{branch}"}).get("value", [])
    return next((r for r in refs if r.get("name") == f"refs/heads/{branch}"), None)


def ado_create_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": "0" * 40,
                                          "newObjectId": commit}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"),
                        f"couldn't create branch {branch}", branch)


def ado_lock(gp, branch, locked=True):
    ado(gp, "PATCH", "/refs", params={"filter": f"heads/{branch}"}, body={"isLocked": locked})


def ado_delete_branch(gp, branch, commit):
    res = ado(gp, "POST", "/refs", body=[{"name": f"refs/heads/{branch}", "oldObjectId": commit,
                                          "newObjectId": "0" * 40}]).get("value", [{}])
    if not res or not res[0].get("success"):
        raise HttpError(None, (res[0] if res else {}).get("updateStatus", "Failed"), f"couldn't delete {branch}", branch)


def ado_default_head(gp):
    """(default branch name, its head commit) of the repository."""
    repo = ado(gp, "GET", "")
    name = (repo.get("defaultBranch") or "refs/heads/main")[len("refs/heads/"):]
    ref = ado_ref(gp, name)
    return name, (ref or {}).get("objectId")


def ado_tree(gp, commit, missing_ok=False):
    """Every file path under the connection's Git folder at a commit ([] if the folder is missing and missing_ok)."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    try:
        body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                                "versionDescriptor.version": commit,
                                                "versionDescriptor.versionType": "commit"})
    except HttpError as exc:
        if missing_ok and exc.status == 404:
            return []
        raise
    return [e["path"] for e in body.get("value", []) if e.get("gitObjectType") == "blob"]


def ado_file(gp, commit, path):
    return ado(gp, "GET", "/items", params={"path": path, "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit", "$format": "octetStream"},
               raw=True)


def git_items_at(gp, commit):
    """Fabric items saved in Git at a commit: {key_of(type, name): {type, displayName, logicalId, folder, files}}."""
    paths = ado_tree(gp, commit)
    out = {}
    for p in paths:
        if not p.endswith("/.platform"):
            continue
        folder = p[: -len("/.platform")]
        try:
            meta = json.loads(ado_file(gp, commit, p).decode("utf-8-sig"))
        except ValueError as exc:
            raise HttpError(None, "BadPlatformFile", f"{p} is not valid JSON: {exc}", p) from None
        md, cfg = meta.get("metadata") or {}, meta.get("config") or {}
        files = [q for q in paths if q.startswith(folder + "/")]
        out[key_of(md.get("type"), md.get("displayName"))] = {
            "type": md.get("type"), "displayName": md.get("displayName"), "logicalId": cfg.get("logicalId"),
            "folder": folder, "files": len(files)}
    return out


# ---------------------------------------------------------------- Fabric Git integration
FOLDER_MARKER = "README.md"     # placeholder V2_1 writes so the workspace's Git folder exists before linking


def folder_marker_path(folder):
    return "/" + folder.strip("/") + "/" + FOLDER_MARKER


def ado_create_folder(gp, branch, folder, text):
    """Create a Git folder by pushing a small marker file to the branch: the Fabric Git connect API (unlike the
    workspace settings screen) doesn't create a missing folder and fails with GitProviderResourceNotFound.
    Returns the new branch head."""
    ref = ado_ref(gp, branch)
    if not ref:
        raise HttpError(None, "BranchNotFound", f"branch {branch} doesn't exist", branch)
    res = ado(gp, "POST", "/pushes", body={
        "refUpdates": [{"name": f"refs/heads/{branch}", "oldObjectId": ref["objectId"]}],
        "commits": [{"comment": f"Create Git folder {folder} for Fabric Git integration",
                     "changes": [{"changeType": "add", "item": {"path": folder_marker_path(folder)},
                                  "newContent": {"content": text, "contentType": "rawtext"}}]}]})
    return ((res.get("commits") or [{}])[0]).get("commitId")


def ado_tree_ids(gp, commit):
    """{path: blob objectId} under the connection's Git folder at a commit: compares folder content cheaply."""
    scope = "/" + (gp.get("directoryName") or "").strip("/")
    body = ado(gp, "GET", "/items", params={"scopePath": scope, "recursionLevel": "Full",
                                            "versionDescriptor.version": commit,
                                            "versionDescriptor.versionType": "commit"})
    return {e["path"]: e.get("objectId") for e in body.get("value", []) if e.get("gitObjectType") == "blob"}


def ado_create_tag(gp, name, commit, message):
    """Annotated tag (refs/tags/<name>) on a commit. Returns the tag's commit, or raises."""
    existing = ado(gp, "GET", "/refs", params={"filter": f"tags/{name}"}).get("value", [])
    if any(r.get("name") == f"refs/tags/{name}" for r in existing):
        raise HttpError(None, "TagExists", f"tag {name} already exists", name)
    return ado(gp, "POST", "/annotatedtags", body={"name": name, "taggedObject": {"objectId": commit},
                                                    "message": message})


# ---------------------------------------------------------------- who runs this, and their workspace role
def token_claims(audience="pbi"):
    """Claims of the runner's Entra token (no secret is printed): oid, upn, idtyp ('app' = service principal)."""
    try:
        part = notebookutils.credentials.getToken(audience).split(".")[1]
        return json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
    except Exception:
        return {}


def me():
    c = token_claims()
    return {"oid": str(c.get("oid", "")).lower(), "upn": c.get("upn") or c.get("unique_name") or c.get("appid", ""),
            "tid": str(c.get("tid", "")).lower(),
            "isApp": c.get("idtyp") == "app" or ("appid" in c and "upn" not in c and "unique_name" not in c)}


def _tenant_from_headers(headers):
    tid = headers.get("X-VSS-ResourceTenant") or headers.get("x-vss-resourcetenant")
    if tid is None:
        m = re.search(r"login\.(?:microsoftonline\.com|windows\.net)/([0-9a-fA-F-]{36})",
                      headers.get("WWW-Authenticate") or headers.get("www-authenticate") or "")
        tid = m.group(1) if m else None
    if tid is None:
        return None
    tid = tid.split(",")[0].strip().lower()
    return "" if tid in ("", "00000000-0000-0000-0000-000000000000") else tid


def sp_settings_missing():
    return [n for n in ("ADO_SP_TENANT_ID", "ADO_SP_CLIENT_ID", "ADO_SP_SECRET_NAME")
            if not str(globals().get(n) or "").strip() or "<" in str(globals().get(n))]


def sp_repo_check(org, project, repo):
    """(True, '') if the service principal in ADO_SP_* can read the repository, else (False, error)."""
    url = f"https://dev.azure.com/{quote(org)}/{quote(project)}/_apis/git/repositories/{quote(repo)}"
    try:
        ado_call(url, auth=sp_header())
        return True, ""
    except HttpError as exc:
        return False, exc.short()


def ado_org_tenant_info(org, project=None, repo=None):
    """(tenant ID, how it was found) of the Entra tenant the DevOps organization is connected to. '' = not connected
    to Entra, None = couldn't tell. 1) Azure DevOps names it in the X-VSS-ResourceTenant (or WWW-Authenticate) header
    of an anonymous call. 2) Otherwise, if the service principal in ADO_SP_* can read the repository, the organization
    is connected to the service principal's tenant: Azure DevOps only accepts service principals of its own tenant."""
    for path in ("_apis/projects?api-version=" + ADO_API_VERSION, "", "_apis/connectionData"):
        try:
            resp = _session().get(f"https://dev.azure.com/{quote(org)}/{path}", timeout=30, allow_redirects=False)
        except requests.RequestException:
            continue
        tid = _tenant_from_headers(resp.headers)
        if tid is not None:
            return tid, "named by Azure DevOps"
    if project and repo and not sp_settings_missing():
        try:
            ok, _ = sp_repo_check(org, project, repo)
        except HttpError:
            ok = False
        if ok:
            return (str(ADO_SP_TENANT_ID).strip().lower(),
                    f"the organization accepts service principal {str(ADO_SP_CLIENT_ID).strip()} of tenant "
                    f"{str(ADO_SP_TENANT_ID).strip()}, and Azure DevOps only accepts service principals of the Entra "
                    "tenant it's connected to")
    return None, ("Azure DevOps didn't name its tenant, and the service principal couldn't confirm it" if project
                  else "Azure DevOps didn't name its tenant")


def ado_org_tenant(org, project=None, repo=None):
    return ado_org_tenant_info(org, project, repo)[0]


def my_workspace_role(ws_id):
    """Your role in a workspace via the admin API (Fabric administrator): 'Admin', 'Member', ..., '' = none,
    None = couldn't tell."""
    who = me()
    try:
        users = fabric("GET", f"/admin/workspaces/{ws_id}/users").get("accessDetails") or []
    except HttpError:
        return None
    for u in users:
        pr = u.get("principal") or {}
        upn = ((pr.get("userDetails") or {}).get("userPrincipalName") or "").lower()
        if str(pr.get("id", "")).lower() == who["oid"] or (upn and upn == str(who["upn"]).lower()):
            return (u.get("workspaceAccessDetails") or {}).get("workspaceRole") or ""
    return ""


def add_me_as_workspace_admin(ws_id):
    """Fabric/Power BI administrator API: add the runner as Admin of a workspace."""
    who = me()
    http("POST", f"https://api.powerbi.com/v1.0/myorg/admin/groups/{ws_id}/users", "pbi",
         json_body={"identifier": who["oid"], "principalType": "App" if who["isApp"] else "User",
                    "groupUserAccessRight": "Admin"}, label="fabric")


CROSS_GEO_TITLE = "Users can export items to Git repositories in other geographical locations"
CROSS_GEO_FIX = ("The workspace's capacity is in another geography than the Azure DevOps organization. A Fabric "
                 "administrator must enable Admin portal > Tenant settings > Git integration > '" + CROSS_GEO_TITLE +
                 "' for the entire organization, or for a security group that includes everyone who runs V2_1 to "
                 "V2_5 (the setting applies to the person doing the Git operation, not to the DevOps account). Only "
                 "item definitions cross geographies, never data. Wait a few minutes after Apply, then rerun: the "
                 "notebook resumes. Linking by hand fails the same way until the setting is on.")


def git_tenant_settings():
    """Git integration tenant settings from the admin API, by title (admin role needed)."""
    res = fabric("GET", "/admin/tenantsettings")
    return {t.get("title", t.get("settingName", "")): t for t in res.get("tenantSettings", [])
            if "git" in (t.get("title", "") + t.get("tenantSettingGroup", "")).lower()}


def git_connection_url(org, project, repo):
    return f"https://dev.azure.com/{quote(org)}/{quote(project)}/_git/{quote(repo)}/"


def find_git_connection(name):
    """Fabric connection with this display name that you can use, or None."""
    return next((c for c in fabric_pages("/connections") if c.get("displayName") == name), None)


def create_git_connection(name, org, project, repo):
    """Fabric cloud connection 'Azure DevOps - Source control' with the service principal in ADO_SP_*. The
    client secret is read from Key Vault and only sent to Fabric."""
    secret = _clean_secret(notebookutils.credentials.getSecret(KEY_VAULT_URL, ADO_SP_SECRET_NAME))
    body = {"displayName": name, "connectivityType": "ShareableCloud",
            "connectionDetails": {"type": "AzureDevOpsSourceControl", "creationMethod": "AzureDevOpsSourceControl.Contents",
                                  "parameters": [{"dataType": "Text", "name": "url",
                                                  "value": git_connection_url(org, project, repo)}]},
            "credentialDetails": {"credentials": {"credentialType": "ServicePrincipal",
                                                  "tenantId": str(ADO_SP_TENANT_ID).strip(),
                                                  "servicePrincipalClientId": str(ADO_SP_CLIENT_ID).strip(),
                                                  "servicePrincipalSecret": secret}}}
    return fabric("POST", "/connections", body)


GIT_LINK_MODES = {"service_principal": "service principal (Fabric connection)",
                  "my_account": "your own Azure DevOps sign-in",
                  "connection_id": "an existing Fabric connection (GIT_CONNECTION_ID)"}


def git_link_mode():
    """How Fabric signs in to Azure DevOps for the workspaces' Git link (GIT_LINK_AUTH)."""
    raw = globals().get("GIT_LINK_AUTH")
    if raw is None:                       # older settings cells
        if globals().get("CREATE_GIT_CONNECTION"):
            return "service_principal"
        return "connection_id" if str(globals().get("GIT_CONNECTION_ID") or "").strip() else "my_account"
    m = str(raw).strip().lower().replace("-", "_").replace(" ", "_")
    m = {"sp": "service_principal", "spn": "service_principal", "serviceprincipal": "service_principal",
         "user": "my_account", "me": "my_account", "automatic": "my_account", "sso": "my_account",
         "connection": "connection_id", "configuredconnection": "connection_id"}.get(m, m)
    if m not in GIT_LINK_MODES:
        raise ValueError(f"GIT_LINK_AUTH = {raw!r}: use \"service_principal\", \"my_account\" or \"connection_id\"")
    return m


def _read_git_connection(cid):
    try:
        c = fabric("GET", f"/connections/{cid}")
    except HttpError as exc:
        raise RuntimeError(f"Can't read Fabric connection {cid} ({exc.short()}): check GIT_CONNECTION_ID, and that you "
                           "have User permission on it (Settings > Manage connections and gateways).") from None
    t = (c.get("connectionDetails") or {}).get("type")
    if t != "AzureDevOpsSourceControl":
        raise RuntimeError(f"Fabric connection {cid} ({c.get('displayName')}) is of type {t}, not Azure DevOps - "
                           "Source control: choose another connection.")
    return c


def plan_git_link(who, org, project, repo, allow_create=True):
    """How Fabric will sign in to Azure DevOps when it links a workspace, from GIT_LINK_AUTH. Returns {mode,
    connectionId, create (name of the connection to create on confirm, or None), text}. Raises before anything
    changes when the choice can't work."""
    mode = git_link_mode()
    cid = str(globals().get("GIT_CONNECTION_ID") or "").strip()
    if mode == "my_account":
        if cid:
            raise ValueError("GIT_CONNECTION_ID is filled but GIT_LINK_AUTH = \"my_account\": empty GIT_CONNECTION_ID, "
                             "or set GIT_LINK_AUTH = \"connection_id\" to use that connection.")
        if who["isApp"]:
            raise RuntimeError("This notebook runs as a service principal / workspace identity, which has no Azure "
                               "DevOps sign-in of its own: set GIT_LINK_AUTH = \"service_principal\".")
        return {"mode": mode, "connectionId": "", "create": None,
                "text": f"your own Azure DevOps sign-in ({who['upn']}). That only works if the DevOps organization "
                        f"is connected to the same Entra tenant as Fabric and {who['upn']} is a member of it, with "
                        "Basic access and project Contributors."}
    if mode == "connection_id":
        if not cid:
            raise ValueError("GIT_LINK_AUTH = \"connection_id\" needs GIT_CONNECTION_ID.")
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"the identity stored in Fabric connection {c.get('displayName')} ({cid}). Give that identity "
                        "Basic access and project Contributors in Azure DevOps."}
    need = sp_settings_missing()
    if need:
        raise ValueError("GIT_LINK_AUTH = \"service_principal\" needs: " + ", ".join(need) +
                         " (fill them even if ADO_AUTH = \"pat\").")
    sp_tid = str(ADO_SP_TENANT_ID).strip().lower()
    if who.get("tid") and sp_tid != who["tid"] and not globals().get("ALLOW_CROSS_TENANT"):
        raise RuntimeError(f"Stopped: the service principal's tenant ({sp_tid}) isn't the Fabric tenant ({who['tid']}). "
                           "Use an app registration from the same tenant (ALLOW_CROSS_TENANT is not part of this "
                           "migration).")
    sp_text = (f"service principal {str(ADO_SP_CLIENT_ID).strip()} (not tied to a person). Fabric stores its client "
               "secret in the connection: when you renew the secret, update the connection too.")
    if cid:
        c = _read_git_connection(cid)
        return {"mode": mode, "connectionId": cid, "create": None,
                "text": f"Fabric connection {c.get('displayName')} ({cid}), which signs in as the " + sp_text}
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    found = find_git_connection(name)
    if found:
        t = (found.get("connectionDetails") or {}).get("type")
        cred = (found.get("credentialDetails") or {}).get("credentialType")
        if t != "AzureDevOpsSourceControl" or (cred and cred != "ServicePrincipal"):
            raise RuntimeError(f"A Fabric connection named {name} already exists but it's {t} / {cred}, not Azure DevOps "
                               "- Source control with a service principal: set another GIT_CONNECTION_NAME.")
        return {"mode": mode, "connectionId": found["id"], "create": None,
                "text": f"existing Fabric connection {name} ({found['id']}), which signs in as the " + sp_text}
    if not allow_create:
        return {"mode": mode, "connectionId": "", "create": name,
                "text": f"Fabric connection {name} (doesn't exist yet), which will sign in as the " + sp_text}
    return {"mode": mode, "connectionId": "", "create": name,
            "text": f"Fabric connection {name}, created when you confirm the plan, which signs in as the " + sp_text}


SAME_TENANT_FIX = (
    "This migration keeps everything in one Entra tenant: the Azure DevOps organization must be connected to the "
    "same Entra tenant as Fabric. Either use a DevOps organization that is already in this tenant, or connect this "
    "one: Azure DevOps > Organization settings > Microsoft Entra > Connect directory (Project Collection "
    "Administrator; existing users must exist in the tenant or they lose access, so plan it with the DevOps admin). "
    "Then add the people who run the notebooks (Basic access, project Contributors), open a workspace > Source "
    "control once and sign in. Linking across tenants (ALLOW_CROSS_TENANT = True) is not part of this migration.")
FOLDER_FIRST = (
    "Most common cause: the Git folder doesn't exist in the branch (the connect API doesn't create it, unlike the "
    "workspace settings screen). V2_1 creates it before linking; if you link by hand or with another tool, create "
    "the folder first (a file inside it, for example README.md). ")
GIT_NOT_FOUND_FIX = (
    "Fabric couldn't see the organization, project, repository or branch with your own Azure DevOps sign-in. This "
    "notebook already found all of them with the PAT/service principal, so the names are right (those settings are "
    "only for the notebook's own DevOps calls). With your own sign-in Fabric only sees DevOps organizations connected "
    "to this Fabric tenant's Entra ID in which you are a member. Check, in this order: (1) you are a member of the "
    "organization with Basic access and in the project's Contributors; (2) the organization is connected to this "
    "Entra tenant (V2_0 row 'organization in the same Entra tenant as Fabric'). " + SAME_TENANT_FIX + " Optional, "
    "still within the same tenant: GIT_LINK_AUTH = \"service_principal\" links with the service principal "
    "instead of your own account.")
GIT_NOT_FOUND_WITH_CONNECTION_FIX = (
    "Fabric couldn't see the repository, branch or folder with the Git connection's identity. " + FOLDER_FIRST +
    "Otherwise: Fabric couldn't see the repository with the Git connection's identity. Give that identity (for example the "
    "service principal) access: Azure DevOps > Organization settings > Users > Add (Basic), and the project's "
    "Contributors group. Check that the connection URL points to this organization/project/repository, and that "
    "the connection's tenant ID is the tenant the DevOps organization is connected to.")


def git_connect_fix(exc):
    """How to fix a failed Git connect / initialize call."""
    text = f"{getattr(exc, 'code', '')} {exc}".lower()
    if "principaltypenotsupported" in text or "service principal" in text:
        return ("The notebook runs as a service principal / workspace identity: Fabric only accepts that with a "
                "stored connection: set GIT_LINK_AUTH = \"service_principal\", or run the notebook interactively as "
                "a user.")
    if "gitproviderresourcenotfound" in text:
        return FOLDER_FIRST + "Otherwise: " + GIT_NOT_FOUND_FIX if git_link_mode() == "my_account" else \
            GIT_NOT_FOUND_WITH_CONNECTION_FIX
    if "insufficientprivileges" in text or getattr(exc, "status", None) == 403:
        return ("You must be Admin of the workspace (Fabric administrator isn't enough). Set "
                "ADD_ME_AS_WORKSPACE_ADMIN = True, or ask a workspace admin to add you.")
    if "credential" in text or "gitprovider" in text or "tf400813" in text or "unauthorized" in text:
        return ("Fabric has no working Azure DevOps credentials for you. Either open any workspace > Source control "
                "once and sign in to Azure DevOps (same Entra tenant as the DevOps organization), or set "
                "GIT_LINK_AUTH = \"service_principal\" so Fabric signs in with the service principal instead.")
    if "crossgeo" in text or "cross geo" in text or "cross-geo" in text:
        return CROSS_GEO_FIX
    if "capacity" in text:
        return "Assign the workspace to a Fabric capacity first."
    if "alreadyconnected" in text:
        return "Already connected: rerun the notebook, it will check the existing link."
    if "directory" in text or "folder" in text or "branch" in text:
        return "Check the branch and folder: the folder must be empty or belong to this workspace."
    return "See the error; if it persists, link this workspace by hand with the values in the table."


def current_workspace_id():
    """Workspace this notebook runs in ('' if unknown)."""
    try:
        ctx = notebookutils.runtime.context
        return str(ctx.get("currentWorkspaceId") or "").lower()
    except Exception:
        return ""


RUN_FROM_ELSEWHERE = ("Run the migration notebooks from a separate workspace that isn't migrated (for example "
                      "'fabric-migration-control'): this workspace is in scope, so its notebooks, including the one "
                      "running, would be deleted with the other items.")


def my_git_credentials(ws_id):
    """Your own Git credentials for a linked workspace ({'source': 'Automatic' | 'ConfiguredConnection' | 'None',
    'connectionId'}). Per user and per workspace: the workspace screens (Source control, Git integration) use them."""
    try:
        return fabric("GET", f"/workspaces/{ws_id}/git/myGitCredentials")
    except HttpError as exc:
        return {"source": "?", "error": exc.short()}


def creds_need_fix(cred, connection_id):
    """True if your Git credentials for the workspace should be switched to the Fabric connection."""
    if git_link_mode() == "my_account":
        return False
    return cred.get("source") != "ConfiguredConnection" or bool(connection_id and cred.get("connectionId") != connection_id)


def set_my_git_credentials(ws_id, connection_id):
    fabric("PATCH", f"/workspaces/{ws_id}/git/myGitCredentials",
           {"source": "ConfiguredConnection", "connectionId": connection_id})


CREDS_WHY = ("your own Git sign-in for this workspace is '{src}', so the workspace screens (Source control, Git "
             "integration) try your personal Azure DevOps account and show 'Can't fetch branches' / 'connection "
             "removed'. The link itself is fine.")


def git_connection_label():
    """Name of the Fabric connection used to link (GIT_CONNECTION_NAME or the default name)."""
    if git_link_mode() == "connection_id":
        return f"with ID {globals().get('GIT_CONNECTION_ID')}"
    return (str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{globals().get('ADO_ORGANIZATION')}"
            f"-{globals().get('ADO_PROJECT')}-{globals().get('ADO_REPOSITORY')}")


def git_creds_hint(ws_id, gp=None):
    """Plain-language fix for a Git call (status, commit, update, disconnect) that fails because of YOUR OWN Git
    sign-in for the workspace. The notebooks don't change that sign-in by themselves: you pick it in the workspace."""
    cred = my_git_credentials(ws_id)
    gp = gp or {}
    org, project, repo = (gp.get("organizationName") or globals().get("ADO_ORGANIZATION", ""),
                          gp.get("projectName") or globals().get("ADO_PROJECT", "<project>"),
                          gp.get("repositoryName") or globals().get("ADO_REPOSITORY", "<repository>"))
    name = str(globals().get("GIT_CONNECTION_NAME") or "").strip() or f"fabric-migration-{org}-{project}-{repo}"
    src = cred.get("source")
    now = f"Your Git sign-in for this workspace is '{src}'" + (
        f" (connection {cred.get('connectionId')})" if cred.get("connectionId") else "") + "."
    return (f"{now} Git calls run with YOUR OWN Git sign-in for each workspace. Open the workspace > Source control "
            f"> account/credentials and choose the Fabric connection '{name}' (the one used to link it; if it isn't "
            "listed, ask its owner to share it with you: Manage connections and gateways > Manage users), then rerun. "
            "If the workspace was linked with your own account (GIT_LINK_AUTH = \"my_account\"), sign in to Azure "
            "DevOps there instead.")


def git_fail_text(exc, ws_id, gp=None):
    """Full error text for a failed Fabric Git call plus the fix: role (403) or your own Git sign-in (other errors)."""
    text = exc.short()
    if "/git/" not in str(getattr(exc, "url", "") or ""):
        return text
    if exc.status == 403:
        return (text + " | FIX: Git calls need Contributor (status, commit, update) or Admin (connect, disconnect) on "
                "the workspace for the account running the notebook.")
    try:
        return text + " | FIX: " + git_creds_hint(ws_id, gp)
    except Exception:          # the hint is best effort; never hide the original error
        return text


def git_connection(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/connection")


def git_status(ws_id):
    return fabric("GET", f"/workspaces/{ws_id}/git/status")


def resolve_target_workspace(old_id, name, override=None, recorded=None):
    """(workspace ID after the move, how it was found) or (None, why not). A home-region move keeps the workspace
    IDs, so the same ID is tried first; if it's gone, the workspace with the same name (names are unique in a
    tenant). override = WORKSPACE_MAP entry; recorded = the target V2_5 already used (restore_state.json)."""
    if override:
        return override, "WORKSPACE_MAP"
    if recorded:
        return recorded, "same as V2_5" + ("" if recorded == old_id else " (ID changed)")
    try:
        fabric("GET", f"/workspaces/{old_id}")
        return old_id, "same ID"
    except HttpError as exc:
        if exc.status not in (403, 404):
            raise
    try:
        found = [w for w in fabric_pages("/workspaces") if (w.get("displayName") or "") == name]
    except HttpError as exc:
        return None, f"workspace {old_id} not found and the workspace list can't be read ({exc.short()})"
    if len(found) == 1:
        return norm_id(found[0]["id"]), f"found by name; ID changed from {old_id}"
    if not found:
        return None, (f"workspace {old_id} ({name}) not found, and no workspace you can access has that name. "
                      "Recreate it or add yourself to it; if it was renamed, set WORKSPACE_MAP")
    return None, f"workspace {old_id} not found and {len(found)} workspaces are named {name}: set WORKSPACE_MAP"


def workspace_items(ws_id):
    return [{"id": i["id"], "type": i["type"], "displayName": i["displayName"], "workspaceId": ws_id}
            for i in fabric_pages(f"/workspaces/{ws_id}/items")]


# ---------------------------------------------------------------- migration records (ADLS)
def migration_prefix(snapshot_id):
    return f"{ADLS_ROOT.strip('/')}/_migration/{snapshot_id}"


def plan_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


def load_manifest(snapshot_id):
    """The manifest written by the snapshot notebook, checked against its stored hash."""
    if not snapshot_id or "<" in snapshot_id:
        raise ValueError("Set SNAPSHOT_ID to the snapshot ID printed by the snapshot notebook.")
    prefix = migration_prefix(snapshot_id)
    try:
        raw = adls_read(f"{prefix}/manifest.json")
        stored = adls_read(f"{prefix}/manifest.sha256").decode().strip()
    except HttpError as exc:
        raise RuntimeError(f"Can't read the snapshot manifest under {prefix}/ ({exc.short()}).") from None
    if hashlib.sha256(raw).hexdigest() != stored:
        raise RuntimeError("The manifest doesn't match its stored hash: it was changed after the snapshot. Stop and "
                           "investigate; take a new snapshot.")
    return json.loads(raw)


def save_log(prefix, name):
    df = pd.DataFrame(RESULTS, columns=["Workspace", "Item", "Type", "ItemId", "Step", "Status", "Detail"])
    adls_write(f"{prefix}/{name}_{NOW_ID}.csv", df.to_csv(index=False))
    return df

## 1. Current connections and plan

In [ ]:
ADO_ORGANIZATION = ado_org_name(ADO_ORGANIZATION)   # a URL works too
missing = [n for n in ("ADO_ORGANIZATION", "ADO_PROJECT", "ADO_REPOSITORY", "KEY_VAULT_URL")
           if "<" in str(globals()[n])] + ado_auth_missing()
if missing:
    raise ValueError("Fill in these settings first: " + ", ".join(missing))
ws_ids = [norm_id(w) for w in WORKSPACE_IDS]
if not ws_ids or None in ws_ids:
    raise ValueError("WORKSPACE_IDS must list at least one workspace ID (GUID).")
WHO = me()
if current_workspace_id() in ws_ids:
    print("NOTE: this notebook runs inside a workspace it links. Its notebooks will be saved in Git and deleted by "
          "V2_4 with the other items. " + RUN_FROM_ELSEWHERE)
print(f"Running as {WHO['upn'] or '(unknown)'}" + (" (service principal)" if WHO["isApp"] else ""))
LINK = plan_git_link(WHO, ADO_ORGANIZATION, ADO_PROJECT, ADO_REPOSITORY)
GIT_CONNECTION_ID, CREATE_CONNECTION = LINK["connectionId"], LINK["create"]
print(f"Git link method (GIT_LINK_AUTH = \"{LINK['mode']}\"): Fabric will sign in to Azure DevOps with {LINK['text']}")
print(f"(ADO_AUTH = \"{ado_mode()}\" is only how this notebook itself reads and writes in DevOps.)")
REPO = {"organizationName": ADO_ORGANIZATION, "projectName": ADO_PROJECT, "repositoryName": ADO_REPOSITORY}
try:
    default_branch, default_head = ado_default_head(REPO)
    ref = ado_ref(REPO, ADO_BRANCH)
except HttpError as exc:
    raise RuntimeError(f"Can't read repository {ADO_ORGANIZATION}/{ADO_PROJECT}/{ADO_REPOSITORY} ({exc.short()}). "
                       "Check the names, the Key Vault secret and the token's Code scope.") from None
BRANCH_HEAD = (ref or {}).get("objectId")
if not BRANCH_HEAD and not CREATE_BRANCH_IF_MISSING:
    raise RuntimeError(f"Branch {ADO_BRANCH} doesn't exist and CREATE_BRANCH_IF_MISSING is False.")
print(f"Checked in DevOps: repository {ADO_REPOSITORY} found; branch {ADO_BRANCH} " +
      (f"exists (latest commit {BRANCH_HEAD[:8]})" if BRANCH_HEAD else
       f"doesn't exist yet: it will be created from {default_branch} when you confirm"))
if LINK["mode"] == "service_principal":
    sp_ok, sp_err = sp_repo_check(ADO_ORGANIZATION, ADO_PROJECT, ADO_REPOSITORY)
    if not sp_ok:
        raise RuntimeError(f"Stopped: service principal {str(ADO_SP_CLIENT_ID).strip()} can't read repository "
                           f"{ADO_REPOSITORY} ({sp_err}). Fabric would fail to link with it. " + SP_NOT_IN_ORG)
    print(f"Checked in DevOps: service principal {str(ADO_SP_CLIENT_ID).strip()} can read repository {ADO_REPOSITORY}")
ORG_TENANT, ORG_TENANT_HOW = ado_org_tenant_info(ADO_ORGANIZATION, ADO_PROJECT, ADO_REPOSITORY)
if ORG_TENANT is None:
    same_tenant = (f"couldn't tell which Entra tenant it uses ({ORG_TENANT_HOW}). Check it in Azure DevOps > "
                   "Organization settings > Microsoft Entra; it must be the Fabric tenant " + (WHO.get("tid") or "?"))
elif ORG_TENANT == "":
    same_tenant = "NO: the DevOps organization isn't connected to any Entra tenant"
elif ORG_TENANT == WHO.get("tid"):
    same_tenant = f"yes, same Entra tenant as Fabric ({ORG_TENANT}; {ORG_TENANT_HOW})"
else:
    same_tenant = (f"NO: the DevOps organization uses Entra tenant {ORG_TENANT}, Fabric uses {WHO.get('tid') or '?'} "
                   f"({ORG_TENANT_HOW})")
print(f"DevOps organization {ADO_ORGANIZATION} in the Fabric tenant: {same_tenant}")
if same_tenant.startswith("NO") and not ALLOW_CROSS_TENANT:
    raise RuntimeError(f"Stopped: {same_tenant}. " + SAME_TENANT_FIX)


def folder_for(ws_id, name):
    path = DIRECTORY_MAP.get(ws_id) or DIRECTORY_PATTERN.format(
        workspace=re.sub(r"[^A-Za-z0-9 ._-]+", "_", name).strip() or ws_id, id=ws_id)
    return "/" + path.strip("/")


rows, PLAN, used = [], [], {}
for ws_id in dict.fromkeys(ws_ids):
    try:
        info = fabric("GET", f"/workspaces/{ws_id}")
        conn = git_connection(ws_id)
        items = workspace_items(ws_id)
    except HttpError as exc:
        rows.append({"Workspace": ws_id, "State": "?", "Action": "blocked", "Detail": f"can't read it: {exc.short()}"})
        continue
    name, state = info.get("displayName", ws_id), conn.get("gitConnectionState")
    gp = conn.get("gitProviderDetails") or {}
    row = {"Workspace": name, "WorkspaceId": ws_id, "State": state, "Fabric items": sum(
        i["type"] not in POWER_BI_TYPES for i in items), "Repository": gp.get("repositoryName", ""),
        "Branch": gp.get("branchName", ""), "Folder": gp.get("directoryName", "")}
    if state == "ConnectedAndInitialized":
        same = (str(gp.get("organizationName", "")).lower(), str(gp.get("projectName", "")).lower(),
                str(gp.get("repositoryName", "")).lower()) == (ADO_ORGANIZATION.lower(), ADO_PROJECT.lower(),
                                                                ADO_REPOSITORY.lower())
        cred = my_git_credentials(ws_id)
        row["Your Git sign-in"] = cred.get("source")
        if same and creds_need_fix(cred, GIT_CONNECTION_ID) and SWITCH_MY_GIT_CREDENTIALS:
            row.update(Action="already connected; switch your Git sign-in to the connection",
                       Detail=CREDS_WHY.format(src=cred.get("source")))
            PLAN.append({"id": ws_id, "name": name, "folder": gp.get("directoryName", ""), "resume": True,
                         "addAdmin": False, "credsOnly": True})
        else:
            detail = "" if same else "connected to a different repository: check this is intended"
            if same and creds_need_fix(cred, GIT_CONNECTION_ID):
                detail = ("synced; nothing to do here. If the workspace screens show 'Can't fetch branches': "
                          + CREDS_WHY.format(src=cred.get("source")) + " Fix: Source control > account > choose "
                          f"connection {git_connection_label()}.")
            row.update(Action="already connected (left as is)", Detail=detail)
        used[(gp.get("branchName"), str(gp.get("directoryName", "")).strip("/").lower())] = name
    elif state == "Connected":
        mine = (str(gp.get("repositoryName", "")).lower(), gp.get("branchName"), "/" + str(gp.get(
            "directoryName", "")).strip("/").lower()) == (ADO_REPOSITORY.lower(), ADO_BRANCH,
                                                         folder_for(ws_id, name).lower())
        if mine:
            row.update(Action="finish link (first commit)", Detail="linked earlier but never synced")
            PLAN.append({"id": ws_id, "name": name, "folder": folder_for(ws_id, name), "resume": True,
                         "addAdmin": False})
        else:
            row.update(Action="blocked", Detail="connected elsewhere but never synced: finish it in Source "
                       "control, or disconnect")
    else:
        folder = folder_for(ws_id, name)
        row["Repository"], row["Branch"], row["Folder"] = ADO_REPOSITORY, ADO_BRANCH, folder
        problems = []
        if not info.get("capacityId"):
            problems.append("no capacity assigned")
        base = BRANCH_HEAD or default_head
        existing = ado_tree({**REPO, "directoryName": folder}, base, missing_ok=True) if base else []
        if [x for x in existing if x.lower() != folder_marker_path(folder).lower()]:
            problems.append(f"Git folder {folder} isn't empty")
        if (ADO_BRANCH, folder.strip("/").lower()) in used:
            problems.append(f"folder {folder} is already used by {used[(ADO_BRANCH, folder.strip('/').lower())]}")
        used[(ADO_BRANCH, folder.strip("/").lower())] = name
        role, add_admin = my_workspace_role(ws_id), False
        if role is not None and role != "Admin":
            if ADD_ME_AS_WORKSPACE_ADMIN:
                add_admin = True
            else:
                problems.append(f"you're {role or 'not a member'} here; linking needs Admin (set "
                                "ADD_ME_AS_WORKSPACE_ADMIN = True, or ask a workspace admin)")
        row["Your role"] = "unknown" if role is None else (role or "none")
        if problems:
            row.update(Action="blocked", Detail="; ".join(problems))
        else:
            row.update(Action=("add you as Admin, " if add_admin else "") + ("" if existing else "create Git folder, ")
                       + "connect + first commit", Detail="")
            PLAN.append({"id": ws_id, "name": name, "folder": folder, "resume": False, "addAdmin": add_admin,
                         "createFolder": not existing})
    rows.append(row)

CHECK_DF = pd.DataFrame(rows)
show(CHECK_DF.drop(columns=["WorkspaceId"], errors="ignore"))
CONNECT_ID = plan_hash({"repo": REPO, "branch": ADO_BRANCH, "head": BRANCH_HEAD, "plan": PLAN,
                        "linkMode": LINK["mode"], "connection": GIT_CONNECTION_ID, "createConnection": CREATE_CONNECTION})
print(f"\n{len(PLAN)} workspace(s) to connect. CONNECT ID: {CONNECT_ID}")

## 2. Connect and first commit

In [ ]:
if DRY_RUN:
    print(f"DRY_RUN = True: nothing changed. To connect, set DRY_RUN = False and CONFIRM_CONNECT_ID = \"{CONNECT_ID}\".")
elif PLAN and CONFIRM_CONNECT_ID != CONNECT_ID:
    raise RuntimeError(f"CONFIRM_CONNECT_ID doesn't match this plan's CONNECT ID ({CONNECT_ID}). Review the plan above "
                       "and confirm again.")
elif PLAN:
    if CREATE_CONNECTION:
        try:
            GIT_CONNECTION_ID = create_git_connection(CREATE_CONNECTION, ADO_ORGANIZATION, ADO_PROJECT, ADO_REPOSITORY)["id"]
            log("(all)", {}, "Git connection", "OK", f"created {CREATE_CONNECTION} ({GIT_CONNECTION_ID})")
        except HttpError as exc:
            raise RuntimeError(f"Couldn't create the Fabric connection {CREATE_CONNECTION}: {exc.short()}. Check the "
                               "service principal settings and its client secret; if a connection with this name "
                               "exists but isn't shared with you, ask its owner to share it (Manage connections and "
                               "gateways > Manage users) or set GIT_CONNECTION_NAME to a new name. Nothing was "
                               "linked.") from None
        CREATE_CONNECTION = None
    if not BRANCH_HEAD:
        ado_create_branch(REPO, ADO_BRANCH, default_head)
        print(f"Created branch {ADO_BRANCH} from {default_branch}")
    for p in PLAN:
        try:
            if p["addAdmin"]:
                add_me_as_workspace_admin(p["id"])
                log(p["name"], {}, "Workspace role", "OK", f"{WHO['upn']} added as Admin (remove after the migration)")
            if p.get("credsOnly"):
                set_my_git_credentials(p["id"], GIT_CONNECTION_ID)
                log(p["name"], {}, "Git sign-in", "OK", f"your Git credentials now use connection {GIT_CONNECTION_ID}")
                continue
            if p.get("createFolder"):
                try:
                    head = ado_create_folder(REPO, ADO_BRANCH, p["folder"],
                                             f"# {p['name']}\n\nGit folder of Fabric workspace {p['name']} ({p['id']}), "
                                             "created by V2_1 for the tenant home-region migration. Fabric item "
                                             "folders are added next to this file by Git integration.\n")
                except HttpError as exc:
                    log(p["name"], {}, "Git folder", "Failed", f"{exc.short()} | FIX: the notebook's DevOps sign-in "
                        f"(ADO_AUTH) needs Contribute on {ADO_REPOSITORY}; or create {p['folder']} by hand (any file "
                        "inside it) and rerun.")
                    continue
                p["createFolder"] = False
                log(p["name"], {}, "Git folder", "OK", f"created {p['folder']} on {ADO_BRANCH} ({(head or '')[:8]})")
            body = {"gitProviderDetails": {"gitProviderType": "AzureDevOps", **REPO, "branchName": ADO_BRANCH,
                                           "directoryName": p["folder"]}}
            if GIT_CONNECTION_ID:
                body["myGitCredentials"] = {"source": "ConfiguredConnection", "connectionId": GIT_CONNECTION_ID}
            if not p["resume"]:
                fabric("POST", f"/workspaces/{p['id']}/git/connect", body)
            init = fabric("POST", f"/workspaces/{p['id']}/git/initializeConnection",
                          {"initializationStrategy": "PreferWorkspace"})
            action = init.get("requiredAction")
            if action == "CommitToGit":
                fabric("POST", f"/workspaces/{p['id']}/git/commitToGit",
                       {"mode": "All", "workspaceHead": init.get("workspaceHead"),
                        "comment": f"Initial commit of workspace {p['name']}"}, want_result=False)
            elif action == "UpdateFromGit":
                raise HttpError(None, "UnexpectedAction", "Fabric wants to update the workspace from Git; stopped "
                                "without updating. Check the folder in the repository.", p["id"])
            log(p["name"], {}, "Connect", "OK", f"{ADO_BRANCH}:{p['folder']} ({action})")
            if GIT_CONNECTION_ID and creds_need_fix(my_git_credentials(p["id"]), GIT_CONNECTION_ID) \
                    and not SWITCH_MY_GIT_CREDENTIALS:
                log(p["name"], {}, "Git sign-in", "Info", "linked. If the workspace screens show 'Can't fetch "
                    f"branches': Source control > account > choose connection {git_connection_label()}"
                    " (the notebook doesn't change your Git sign-in unless SWITCH_MY_GIT_CREDENTIALS = True).")
            elif GIT_CONNECTION_ID and creds_need_fix(my_git_credentials(p["id"]), GIT_CONNECTION_ID):
                try:
                    set_my_git_credentials(p["id"], GIT_CONNECTION_ID)
                    log(p["name"], {}, "Git sign-in", "OK", f"your Git credentials use connection {GIT_CONNECTION_ID}")
                except HttpError as exc:
                    log(p["name"], {}, "Git sign-in", "Warning", f"{exc.short()} | FIX: linked, but the workspace "
                        "screens may show 'Can't fetch branches': Source control > account > choose connection "
                        f"{GIT_CONNECTION_ID}, or rerun this notebook.")
        except HttpError as exc:
            log(p["name"], {}, "Connect", "Failed", f"{exc.short()} | FIX: {git_connect_fix(exc)}")
    failed = [p for p in PLAN if any(r["Workspace"] == p["name"] and r["Step"] == "Connect" and r["Status"] == "Failed"
                                     for r in RESULTS)]
    if failed:
        print("\nNot linked. After fixing the cause, rerun this notebook (it resumes). To link by hand instead: "
              "Workspace settings > Git integration > Azure DevOps, with:")
        print("(The Git folder must exist in the branch before an API link; the settings screen creates it for you.)")
        show(pd.DataFrame([{"Workspace": p["name"], "Organization": ADO_ORGANIZATION, "Project": ADO_PROJECT,
                            "Repository": ADO_REPOSITORY, "Branch": ADO_BRANCH, "Git folder": p["folder"]}
                           for p in failed]))
        print("then Source control > Commit all. Rerun this notebook to verify.")
if GIT_CONNECTION_ID:
    print(f"\nGit connection used: {GIT_CONNECTION_ID}. V2_5 (restore) uses the same one with the same GIT_LINK_AUTH "
          f"and GIT_CONNECTION_NAME (or set GIT_CONNECTION_ID = \"{GIT_CONNECTION_ID}\" there).")

## 3. What is in Git now
Every workspace in scope: connection state, uncommitted changes, and Fabric items that are **not** in Git.

In [ ]:
rows = []
for ws_id in dict.fromkeys(ws_ids):
    try:
        info = fabric("GET", f"/workspaces/{ws_id}")
        conn = git_connection(ws_id)
        if conn.get("gitConnectionState") != "ConnectedAndInitialized":
            rows.append({"Workspace": info.get("displayName", ws_id), "State": conn.get("gitConnectionState")})
            continue
        gp = conn["gitProviderDetails"]
        st = git_status(ws_id)
        saved = git_items_at(gp, st["remoteCommitHash"])
        not_in_git = [f"{i['displayName']} ({i['type']})" for i in workspace_items(ws_id)
                      if i["type"] not in CHILD_TYPES and key_of(i["type"], i["displayName"]) not in saved]
        rows.append({"Workspace": info.get("displayName", ws_id), "State": "ConnectedAndInitialized",
                     "Branch": gp.get("branchName"), "Folder": gp.get("directoryName"),
                     "Items in Git": len(saved), "Uncommitted changes": len(st.get("changes") or []),
                     "Not in Git": ", ".join(not_in_git)})
    except HttpError as exc:
        rows.append({"Workspace": ws_id, "State": f"error: {git_fail_text(exc, ws_id)}"})
VERIFY_DF = pd.DataFrame(rows)
show(VERIFY_DF)
if RESULTS:
    show(pd.DataFrame(RESULTS))
print("Items listed under 'Not in Git' rely on the ADLS copies (V2_2 backup and the snapshot's raw copy).")